#Fine-tuning DistilBERT for POS Tagging and Chunking

In [ ]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)

# Task 1: Dataset Selection
# load the CoNLL-2003 dataset.
dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")

# We will focus on Chunking for this training pipeline.
label_list = dataset["train"].features["chunk_tags"].feature.names
print("Label Categories (Chunk Tags):", label_list)

# Task 2: Data Preprocessing & Label Alignment
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    # Tokenize words using BERT tokenizer
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Handle special tokens with -100 so they are ignored in the loss function
            if word_idx is None:
                label_ids.append(-100)
            # Assign the label to the first token of a given word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Handle subwords: assign -100 to subsequent subword tokens
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply preprocessing to the entire dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# Task 3: Model Setup
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Task 5: Evaluation Metric Setup
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove -100 from labels and predictions before evaluation
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


# Task 4: Training Setup
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(2000)), # Subset for faster testing
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
print("Starting training...")
trainer.train()

# Task 6: Inference
# Predict on custom sentences using the Hugging Face pipeline
token_classifier = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

input_text = "John works at Google in California"
predictions = token_classifier(input_text)

print(f"\nInference for: '{input_text}'")
for entity in predictions:
    print(f"Word: {entity['word']} | Chunk Tag: {entity['entity_group']} | Score: {entity['score']:.4f}")

Label Categories (Chunk Tags): ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


C:\Users\ADARSH JAISWAL\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.370780,0.820125,0.820428,0.820276,0.909474
2,No log,0.305299,0.860142,0.845870,0.852947,0.927018
3,No log,0.298402,0.863312,0.847714,0.855442,0.927719


C:\Users\ADARSH JAISWAL\AppData\Local\Programs\Python\Python314\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\ADARSH JAISWAL\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\ADARSH JAISWAL\AppData\Local\Programs\Python\Python314\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Inference for: 'John works at Google in California'
Word: john | Chunk Tag: NP | Score: 0.9755
Word: works | Chunk Tag: VP | Score: 0.8386
Word: at | Chunk Tag: PP | Score: 0.9295
Word: google | Chunk Tag: NP | Score: 0.8964
Word: in | Chunk Tag: PP | Score: 0.9646
Word: california | Chunk Tag: NP | Score: 0.9459


## Differences Between POS Tagging and Chunking
### POS Tagging (Grammar-Level Tagging)

Part-of-Speech (POS) tagging is a fundamental NLP task that assigns grammatical categories such as noun, verb, adjective, etc., to each individual word in a sentence. It focuses on identifying the syntactic role of each token independently. Since it works at the word level, it is generally considered a simpler task.

### Chunking (Phrase-Level Grouping)

Chunking, also known as shallow parsing, is a more advanced task that groups words into meaningful phrases such as noun phrases (NP) or verb phrases (VP). Unlike POS tagging, chunking considers relationships between multiple words and identifies phrase boundaries, making it moderately more complex.

 ## Challenges Faced and Observations
### Subword Tokenization

Transformer models like BERT use WordPiece tokenization, which breaks unknown or rare words into smaller subword units. The main challenge was ensuring correct alignment between original word-level labels and tokenized subwords.

### Handling Special Tokens

Special tokens such as [CLS] and [SEP], along with additional subword tokens, do not correspond to actual labels. Therefore, assigning a value of -100 to these tokens was necessary so that they are ignored during loss computation by the model.

### Evaluation Metrics

Using simple accuracy for sequence labeling tasks can be misleading because the majority class (often "O" – Outside) dominates the dataset. To obtain a more meaningful evaluation, the seqeval library was used to compute Precision, Recall, and F1 Score, which better reflect the model’s performance on actual entities and chunks.